# UIT DSC 2026 — LegalQA v9.2 trên Kaggle T4
Luồng VRAM tuần tự: Embedding/global dense + official train-memory → unload → Reranker → unload → Qwen base hoặc QLoRA adapter → unload → ViLegalBERT → submission.zip. Độ dài được học từ official train; BARTpho không thuộc runtime.

Public inference bắt buộc completed adapter `full`, tuned selector và global dense index. Official train-memory là tùy chọn và phải giữ nguyên giá trị giữa các session resume. Runner dừng an toàn ở 11,5 giờ; nếu cell cuối báo `PAUSED SAFELY`, lưu recovery ZIP và resume ở session kế tiếp.

In [ ]:
DEVICE = 'cuda:0'
USE_FROZEN_V5007_SELECTOR = True  # Baseline public METEOR 0.5007; bỏ qua tuned_selection.yaml attach ngoài
EMBEDDING_BATCH_SIZE = 2
RERANKER_BATCH_SIZE = 1
VERIFIER_BATCH_SIZE = 4
RERANKER_MAX_LENGTH = 512
MAX_CONTEXT_CHARS = 18000
MAX_NEW_TOKENS = 900
BM25_TOP_K = 140
DENSE_TOP_K = 70
FUSED_TOP_K = 48
RERANK_TOP_K = 24
EVIDENCE_TOP_K = 5
FORCE_INDEX = False
REUSE_RETRIEVAL = True
MODEL_SMOKE = True     # Test thật 4 model runtime; BARTpho đã bị loại
FORCE_DENSE_INDEX = False
DENSE_INDEX_ONLY = False  # True cho session 1; session 2 attach index rồi đổi False
DENSE_INDEX_DIR = '/kaggle/working/legalqa_dense_index'
TRAIN_MEMORY = False      # Giữ False khi dense Dataset chỉ có global index; phải khớp session tạo recovery
CHECKPOINT_EVERY = 50  # Lưu recovery ZIP sau mỗi 50 câu mới
AUTO_RESUME = True     # Tự tìm recovery ZIP hoặc thư mục Kaggle đã giải nén
RECOVERY_OUTPUT = '/kaggle/working/legalqa_recovery_latest.zip'
MAX_RUNTIME_HOURS = 11.5  # Dừng trước giới hạn 12 giờ; còn khoảng 30 phút để lưu recovery/output

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
input_root = Path('/kaggle/input')
def require_kaggle_cuda():
    nvidia_smi = shutil.which('nvidia-smi')
    if nvidia_smi is None:
        raise RuntimeError('Session Kaggle chưa được gắn GPU. Vào Settings/Session options → Accelerator → GPU (T4 x2 hoặc T4 x1), restart session rồi Run All.')
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
        raise RuntimeError('Kaggle có nvidia-smi nhưng PyTorch không thấy CUDA. Hãy restart session sau khi bật GPU.')
    subprocess.run([nvidia_smi, '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
    print(f'CUDA OK: {torch.cuda.device_count()} GPU; pipeline dùng cuda:0')
require_kaggle_cuda()
public_files = sorted(input_root.rglob('public-official.json'))
if len(public_files) != 1:
    raise FileNotFoundError(f'Cần duy nhất một public-official.json: {public_files}')
data_root = public_files[0].parent
train_file = data_root / 'train.json'
if not train_file.is_file():
    raise FileNotFoundError(f'Thiếu official train.json cạnh public: {train_file}')
corpus_dirs = [data_root / 'selected-contexts'] if (data_root / 'selected-contexts').is_dir() else []
if not corpus_dirs:
    context_files = list(data_root.rglob('context_*.json'))
    parent_counts = {}
    for path in context_files:
        parent_counts[path.parent] = parent_counts.get(path.parent, 0) + 1
    corpus_dirs = [max(parent_counts, key=parent_counts.get)] if parent_counts else []
if len(corpus_dirs) != 1:
    raise FileNotFoundError(f'Không xác định được corpus: {corpus_dirs}')
EXPECTED_SOURCE_CONTRACT = {
    'schema': 'legalqa-source-contract-v1',
    'release': '0.9.2',
    'agentic_namespace': 'agentic-v3',
    'cache_schema': 'inference-v7-agentic-error-isolation',
    'selector_schema': 'selector-weights-v3-agentic-evidence',
    'adapter_alias_policy': 'sha256-dedupe-v1',
    'source_alias_policy': 'sha256-dedupe-v1',
    'qlora_recovery_schema': 'legalqa-qlora-recovery-v1',
    'inference_pause_schema': 'inference-runtime-pause-v1',
    'offline_wheel_profile': 'hf-stack-2026-08-v1',
    'offline_wheel_count': 12,
}
REQUIRED_WHEELS = ('transformers-5.14.1-', 'tokenizers-0.22.2-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'safetensors-0.8.0-', 'accelerate-1.14.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentence_transformers-5.7.0-', 'sentencepiece-0.2.1-', 'regex-2026.7.19-', 'click-8.4.2-')
def source_contract_errors(path):
    errors = []
    marker = path / 'SOURCE_CONTRACT.json'
    try:
        contract = json.loads(marker.read_text(encoding='utf-8'))
    except Exception as exc:
        return [f'thiếu/hỏng SOURCE_CONTRACT.json: {exc}']
    for key, expected in EXPECTED_SOURCE_CONTRACT.items():
        if contract.get(key) != expected:
            errors.append(f'{key}={contract.get(key)!r}, cần {expected!r}')
    required_files = (path / 'kaggle' / 'run_selector_calibration.py', path / 'kaggle' / 'run_inference_only.py', path / 'src' / 'legalqa' / 'selector_schema.py', path / 'RELEASE_V9_2.md')
    errors.extend(f'thiếu {file.relative_to(path)}' for file in required_files if not file.is_file())
    wheels = list((path / 'wheelhouse').glob('*.whl'))
    names = [wheel.name for wheel in wheels]
    missing_wheels = [prefix for prefix in REQUIRED_WHEELS if not any(name.startswith(prefix) for name in names)]
    if len(wheels) != 12 or missing_wheels or any(wheel.stat().st_size < 10_000 for wheel in wheels):
        errors.append(f'wheelhouse không hợp lệ: count={len(wheels)}, missing={missing_wheels}')
    return errors
def sha256_path(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def source_identity(path):
    digest = hashlib.sha256()
    roots = ('SOURCE_CONTRACT.json', 'pyproject.toml', 'RELEASE_V9_2.md', 'configs', 'kaggle', 'scripts', 'src', 'tests', 'wheelhouse')
    files = []
    for name in roots:
        item = path / name
        files.extend([item] if item.is_file() else ([p for p in item.rglob('*') if p.is_file() and '__pycache__' not in p.parts and '.ipynb_checkpoints' not in p.parts and p.suffix != '.pyc'] if item.is_dir() else []))
    for file in sorted(files, key=lambda p: p.relative_to(path).as_posix()):
        digest.update(file.relative_to(path).as_posix().encode('utf-8')); digest.update(b'\0')
        with file.open('rb') as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml')})
source_reports = {str(path): source_contract_errors(path) for path in source_projects}
qualified_sources = [path for path in source_projects if not source_reports[str(path)]]
source_groups = {}
for path in qualified_sources:
    source_groups.setdefault(source_identity(path), []).append(path)
if len(source_groups) != 1:
    raise RuntimeError(f'Cần đúng một source v9.2 reviewed; source cũ nhúng trong adapter sẽ bị loại. qualified={qualified_sources}; reports={source_reports}')
source_aliases = next(iter(source_groups.values()))
selected_source = min(source_aliases, key=lambda p: (len(p.parts), p.as_posix()))
if len(source_aliases) > 1: print('Source aliases share one SHA-256 identity:', source_aliases)
print('Source v9.2 reviewed:', selected_source)
print('Rejected source copies:', {path: reasons for path, reasons in source_reports.items() if reasons})
source_dir = Path('/kaggle/working/legalqa_source')
if source_dir.exists():
    shutil.rmtree(source_dir)
shutil.copytree(selected_source, source_dir)
project = source_dir
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
required = ('transformers-5.14.1-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'click-8.4.2-', 'regex-2026.7.19-', 'tokenizers-0.22.2-', 'accelerate-1.14.0-', 'safetensors-0.8.0-', 'sentence_transformers-5.7.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentencepiece-0.2.1-')
missing = [prefix for prefix in required if not any(Path(wheel).name.startswith(prefix) for wheel in wheels)]
if missing:
    raise FileNotFoundError(f'Wheelhouse offline chưa đầy đủ: {missing}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
src_path = str(project / 'src')
sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Môi trường source/offline wheels đã sẵn sàng.')
print('Public:', public_files[0])
print('Train:', train_file)
print('Corpus:', corpus_dirs[0])
print('Model configs:', len(list(input_root.rglob('config.json'))))
print('Source:', selected_source)

In [ ]:
dense_index_candidates = {}
for manifest_path in input_root.rglob('manifest.json'):
    try:
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    except Exception:
        continue
    if payload.get('schema') == 'global-dense-v1' and payload.get('status') == 'complete':
        identity = payload.get('identity')
        if not identity: raise RuntimeError(f'Dense manifest completed missing identity: {manifest_path}')
        dense_index_candidates.setdefault(identity, []).append(manifest_path.parent)
if len(dense_index_candidates) > 1:
    raise RuntimeError(f'Chỉ attach tối đa một global dense index: {dense_index_candidates}')
dense_aliases = next(iter(dense_index_candidates.values())) if dense_index_candidates else []
dense_index_input = min(dense_aliases, key=lambda p: (len(p.parts), p.as_posix())) if dense_aliases else None
adapter_candidates = sorted({p.parent for p in input_root.rglob('training_complete.json') if (p.parent / 'adapter_config.json').is_file()})
generator_adapter = adapter_candidates[0] if len(adapter_candidates) == 1 else None
if len(adapter_candidates) > 1: print('Multiple adapters found; runner will enforce scope=full and compare SHA-256:', adapter_candidates)
weight_candidates = sorted(input_root.rglob('tuned_selection.yaml'))
weight_groups = {}
for path in weight_candidates:
    weight_groups.setdefault(sha256_path(path), []).append(path)
frozen_selector = project / 'configs' / 'selector_v5007.yaml'
if USE_FROZEN_V5007_SELECTOR:
    if not frozen_selector.is_file():
        raise FileNotFoundError(f'Source thiếu frozen selector 0.5007: {frozen_selector}')
    selection_weights = frozen_selector
    if weight_candidates:
        print('Baseline 0.5007: bỏ qua tuned_selection.yaml attach ngoài:', weight_candidates)
else:
    if len(weight_groups) > 1:
        raise RuntimeError(f'Multiple tuned_selection.yaml files have DIFFERENT content: {weight_groups}')
    weight_aliases = next(iter(weight_groups.values())) if weight_groups else []
    selection_weights = min(weight_aliases, key=lambda p: (len(p.parts), p.as_posix())) if weight_aliases else None
def recovery_content_identity(path):
    digest = hashlib.sha256()
    if path.is_file():
        with zipfile.ZipFile(path) as archive:
            names = sorted(info.filename for info in archive.infolist() if not info.is_dir())
            for name in names:
                digest.update(name.encode('utf-8')); digest.update(b'\0')
                with archive.open(name) as handle:
                    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
                        digest.update(block)
    else:
        files = sorted((p for p in path.rglob('*') if p.is_file()), key=lambda p: p.relative_to(path).as_posix())
        for file in files:
            digest.update(file.relative_to(path).as_posix().encode('utf-8')); digest.update(b'\0')
            with file.open('rb') as handle:
                for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
                    digest.update(block)
    return digest.hexdigest()
def recovery_state(path):
    try:
        if path.is_dir():
            payload = json.loads((path / 'recovery_state.json').read_text(encoding='utf-8'))
        else:
            with zipfile.ZipFile(path) as archive:
                payload = json.loads(archive.read('recovery_state.json').decode('utf-8'))
    except Exception as exc:
        raise RuntimeError(f'Recovery input không hợp lệ: {path}: {exc}') from exc
    required = ('run_fingerprint', 'stage', 'completed', 'total')
    if not isinstance(payload, dict) or any(key not in payload for key in required):
        raise RuntimeError(f'Recovery state thiếu trường bắt buộc: {path}: {payload}')
    return payload
recovery_inputs = []
if AUTO_RESUME:
    recovery_inputs.extend(sorted(input_root.rglob('legalqa_recovery_latest.zip')))
    expected_questions_sha = sha256_path(public_files[0])
    extracted_roots = set()
    for state_path in input_root.rglob('recovery_state.json'):
        root = state_path.parent
        normalized_name = root.name.lower().replace('-', '_')
        manifests = [root / 'run_manifest.json', root / 'run_manifest.incomplete.json']
        manifest_matches = False
        for manifest_path in manifests:
            if not manifest_path.is_file():
                continue
            try:
                manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
                manifest_matches = (
                    manifest.get('mode') == 'full_sequential_inference_only'
                    and manifest.get('inputs', {}).get('questions_sha256') == expected_questions_sha
                )
            except Exception:
                manifest_matches = False
            if manifest_matches:
                break
        if manifest_matches or (normalized_name == 'legalqa_recovery_latest' and not any(path.is_file() for path in manifests)):
            extracted_roots.add(root)
    recovery_inputs.extend(sorted(extracted_roots))
recovery_groups = {}
recovery_states = {}
for path in recovery_inputs:
    state = recovery_state(path)
    identity = recovery_content_identity(path)
    recovery_groups.setdefault(identity, []).append(path)
    recovery_states[path] = state
if len(recovery_groups) > 1:
    details = {str(path): recovery_states[path] for path in recovery_inputs}
    raise RuntimeError(f'Attach nhiều recovery inference khác nhau; chỉ giữ bản mới nhất: {details}')
recovery_aliases = next(iter(recovery_groups.values())) if recovery_groups else []
resume_bundle = min(recovery_aliases, key=lambda p: (0 if p.is_dir() else 1, len(p.parts), p.as_posix())) if recovery_aliases else None
if resume_bundle is None:
    print('Resume bundle: Không có — chạy mới')
else:
    state = recovery_states[resume_bundle]
    kind = 'extracted directory' if resume_bundle.is_dir() else 'ZIP'
    print(f'Resume bundle: {resume_bundle} [{kind}] stage={state["stage"]} completed={state["completed"]}/{state["total"]} fingerprint={state["run_fingerprint"]}')
print('Global dense index:', dense_index_input or ('Chưa có — session index-only sẽ dựng' if DENSE_INDEX_ONLY else 'THIẾU CHO FINAL INFERENCE'))
print('Qwen adapter:', generator_adapter or ('Runner sẽ tự chọn alias full hợp lệ' if adapter_candidates else 'THIẾU CHO FINAL INFERENCE'))
print('Selector weights:', selection_weights or 'THIẾU CHO FINAL INFERENCE', '(FROZEN 0.5007)' if USE_FROZEN_V5007_SELECTOR else '(EXPERIMENTAL)')
command = [
    sys.executable, str(project / 'kaggle' / 'run_inference_only.py'),
    '--config', str(project / 'configs' / 'kaggle_inference_t4.yaml'),
    '--input-root', '/kaggle/input',
    '--working', '/kaggle/working/legalqa_inference',
    '--submission-output', '/kaggle/working/submission.zip',
    '--recovery-output', RECOVERY_OUTPUT,
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--max-runtime-hours', str(MAX_RUNTIME_HOURS),
    '--device', DEVICE,
    '--embedding-batch-size', str(EMBEDDING_BATCH_SIZE),
    '--reranker-batch-size', str(RERANKER_BATCH_SIZE),
    '--verifier-batch-size', str(VERIFIER_BATCH_SIZE),
    '--reranker-max-length', str(RERANKER_MAX_LENGTH),
    '--max-context-chars', str(MAX_CONTEXT_CHARS),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--bm25-top-k', str(BM25_TOP_K),
    '--dense-top-k', str(DENSE_TOP_K),
    '--fused-top-k', str(FUSED_TOP_K),
    '--rerank-top-k', str(RERANK_TOP_K),
    '--evidence-top-k', str(EVIDENCE_TOP_K),
    '--dense-index-dir', DENSE_INDEX_DIR,
    '--model-smoke' if MODEL_SMOKE else '--no-model-smoke',
    '--reuse-retrieval' if REUSE_RETRIEVAL else '--no-reuse-retrieval',
    '--train-memory' if TRAIN_MEMORY else '--no-train-memory',
    '--auto-resume' if AUTO_RESUME else '--no-auto-resume',
]
if dense_index_input is not None:
    command.extend(['--dense-index-input', str(dense_index_input)])
if resume_bundle is not None:
    command.extend(['--resume-bundle', str(resume_bundle)])
if generator_adapter is not None:
    command.extend(['--generator-adapter', str(generator_adapter)])
if selection_weights is not None:
    command.extend(['--selection-weights', str(selection_weights)])
if DENSE_INDEX_ONLY:
    command.append('--dense-index-only')
else:
    command.extend(['--require-generator-adapter', '--require-selection-weights', '--require-prebuilt-indexes'])
if FORCE_DENSE_INDEX:
    command.append('--force-dense-index')
if FORCE_INDEX:
    command.append('--force-index')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, zipfile
submission = Path('/kaggle/working/submission.zip')
recovery = Path(RECOVERY_OUTPUT)
if DENSE_INDEX_ONLY:
    dense_manifest = Path(DENSE_INDEX_DIR) / 'manifest.json'
    assert dense_manifest.exists(), 'Dense index chưa hoàn tất; state.json/vectors.f16 là checkpoint nội bộ.'
    dense_state = json.loads(dense_manifest.read_text(encoding='utf-8'))
    assert dense_state.get('status') == 'complete'
    if TRAIN_MEMORY:
        memory_manifest = Path(DENSE_INDEX_DIR) / 'train_memory' / 'manifest.json'
        assert memory_manifest.exists(), 'Thiếu completed official train-memory index.'
    print('DENSE INDEX READY:', DENSE_INDEX_DIR, dense_state)
    print('Hãy Save Version, tạo Kaggle Dataset từ thư mục này, attach vào session 2 và đặt DENSE_INDEX_ONLY=False.')
else:
    manifest_path = Path('/kaggle/working/legalqa_inference/run_manifest.json')
    if not manifest_path.is_file():
        raise RuntimeError(f'Thiếu run manifest: {manifest_path}')
    run_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    status = run_manifest.get('status')
    if not recovery.exists():
        raise RuntimeError('Thiếu recovery ZIP sau inference.')
    with zipfile.ZipFile(recovery) as archive:
        state = json.loads(archive.read('recovery_state.json').decode('utf-8'))
    print('RECOVERY:', recovery, state)
    if status == 'paused':
        if submission.exists():
            raise RuntimeError('Run đã paused nhưng vẫn còn submission cũ; không được nộp file này.')
        print('PAUSED SAFELY:', run_manifest.get('pause'))
        print('Save Version, tạo Dataset từ legalqa_recovery_latest.zip, attach vào session mới rồi Run All.')
    elif status == 'complete':
        if not submission.exists():
            raise RuntimeError('Manifest complete nhưng thiếu submission.zip.')
        with zipfile.ZipFile(submission) as archive:
            if archive.namelist() != ['submission.json']:
                raise RuntimeError(f'ZIP chứa sai file: {archive.namelist()}')
            payload = json.loads(archive.read('submission.json').decode('utf-8'))
        official_payload = json.loads(public_files[0].read_text(encoding='utf-8'))
        if set(payload) != set(official_payload):
            raise RuntimeError('question_id trong submission không khớp public-official.json.')
        if not all(isinstance(value, dict) and isinstance(value.get('answer'), str) and value['answer'].strip() for value in payload.values()):
            raise RuntimeError('Submission có answer thiếu/rỗng/không phải string.')
        print(f'READY: {submission} — {submission.stat().st_size:,} bytes — {len(payload)} answers')
    else:
        raise RuntimeError(f'Run status không hợp lệ: {status!r}')